In [ ]:
%%capture
%pip install ultralytics roboflow

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="API-key")
project = rf.workspace("ufvmestrado").project("abelhas-v3")
version = project.version(1)
dataset = version.download("yolo26")

In [ ]:
dataset_paht = "/content/Abelhas-V3-1"
projeto_path = "/content/drive/MyDrive/Lesc-Bee videos/yolo_26n"
treino_name = "modelo_V2_09_02_2026"

In [ ]:
from ultralytics import YOLO

Fine_Tuning = True # Pega o melhor resultado e treinar mais um pouco.
Resume = False # Se False ele inicia um novo treinamento ao inves de continuar de onde parou o ultimo treino.

if Fine_Tuning: # Pega o melhor resultado e treinar mais um pouco.
  model = YOLO(f'{projeto_path}/{treino_name}/weights/best.pt')

  results = model.train(
    data=f'{dataset_paht}/data.yaml',
    cfg="/content/drive/MyDrive/Lesc-Bee videos/modelo_v2/hyp.yaml", # Arquivo com os hiperparametros usados no treino.
    epochs=60,     # Quantas NOVAS épocas você quer rodar
    batch=96,      # Força um lote grande (a T4 aguenta o modelo Nano)
    workers=6,     # Usa todos os núcleos da CPU para preparar as imagens rápido
    cache='disk',    # Carrega as imagens na RAM (RAM Cache). Isso acelera MUITO.
    imgsz=640,
    project=projeto_path,
    name=treino_name # Nome da pasta do experimento
  )
else:
  if Resume: # Continuar em caso de interrupção
    # Carrega o "last.pt" (último estado salvo antes de cair)
    model = YOLO(f'{projeto_path}/{treino_name}/weights/last.pt')
    # Recupera exatamente onde parou (optimizer, learning rate, número da época) e termina as épocas que faltavam.
    results = model.train(resume=True)

  else:
    # Load a model
    model = YOLO('yolo26n.pt')  # load a pretrained model (recommended for training)

    results = model.train(
      data=f'{dataset_paht}/data.yaml',
      cfg="/content/drive/MyDrive/Lesc-Bee videos/modelo_v2/hyp.yaml", # Arquivo com os hiperparametros usados no treino.
      epochs=90,      # Aproximadamente 3h:30min
      batch=96,       # Força um lote grande (a T4 aguenta o modelo Nano)
      workers=6,      # Usa todos os núcleos da CPU para preparar as imagens rápido
      cache='disk',     # Carrega as imagens na RAM (RAM Cache). Isso acelera MUITO.
      imgsz=640,
      project=projeto_path,
      name=treino_name # Nome da pasta do experimento
    )

Realiza o teste do modelo com as imagens de teste

In [ ]:
from ultralytics import YOLO
import os

# --- 1. Carregar o modelo treinado ---
# Certifique-se de que o caminho está correto!
MODEL_PATH = f'{projeto_path}/{treino_name}/weights/best.pt'

# Verifique se o arquivo do modelo existe antes de carregar
if not os.path.exists(MODEL_PATH):
    print(f"Erro: O arquivo do modelo não foi encontrado em: {MODEL_PATH}")
    print("Verifique se o treinamento foi concluído com sucesso e se o caminho está correto.")
else:
    # Carregue o seu melhor modelo
    model = YOLO(MODEL_PATH)
    print("Modelo carregado com sucesso!")

    # --- 2. Executar a validação no conjunto de TESTE ---
    # O arquivo data.yaml contém os caminhos para os conjuntos de treino, validação e teste.
    # Usamos split='test' para forçar a avaliação no conjunto de teste.
    # plots=True irá gerar e salvar os gráficos das métricas.
    print("Iniciando a validação no conjunto de teste...")
    metrics = model.val(
        data=f'{dataset_paht}/data.yaml',
        split='test',
        imgsz=640,
        plots=True,
        project=projeto_path, # Opcional: para organizar os resultados
        name=f'{treino_name}_inference' # Opcional: nome da pasta para esta avaliação
    )
    print("Validação concluída!")

    # --- 3. Acessar e imprimir as métricas ---
    print("\n--- Métricas Gerais ---")
    print(f"mAP50-95 (geral): {metrics.box.map:.4f}")
    print(f"mAP50 (geral):    {metrics.box.map50:.4f}")
    print(f"mAP75 (geral):    {metrics.box.map75:.4f}")

    print("\n--- Métricas de Desempenho no Conjunto de Teste (por classe) ---")

    names = metrics.names  # {id: nome}
    map50_95 = metrics.box.maps
    map50 = metrics.box.all_ap[:, 0]
    map75 = metrics.box.all_ap[:, 5]

    precision = metrics.box.p
    recall = metrics.box.r

    for class_id, class_name in names.items():
        idx = int(class_id)

        print(f"\nClasse: {class_name} (id={idx})")
        print(f"  mAP50-95: {map50_95[idx]:.4f}")
        print(f"  mAP50:    {map50[idx]:.4f}")
        print(f"  mAP75:    {map75[idx]:.4f}")
        print(f"  Precision:{precision[idx]:.4f}")
        print(f"  Recall:   {recall[idx]:.4f}")

In [ ]:
from ultralytics import YOLO
import os
import glob # Biblioteca para encontrar arquivos que correspondem a um padrão
import random # Importe a biblioteca random

# --- 1. Definir os caminhos ---
MODEL_PATH = f'{projeto_path}/{treino_name}/weights/best.pt'
TEST_IMAGES_DIR = f'{dataset_paht}/test/images'
NUM_IMAGES_TO_SELECT = 80 # Defina o número de imagens desejado

# --- 2. Encontrar todas as imagens e selecionar uma amostra aleatória ---
# Use um padrão para encontrar todos os tipos de imagem comuns
padrao_imagens = os.path.join(TEST_IMAGES_DIR, '*.jpg') # Você pode adicionar mais formatos se precisar
lista_completa_imagens = glob.glob(padrao_imagens)

# Verifique se alguma imagem foi encontrada
if not lista_completa_imagens:
    print(f"Nenhuma imagem encontrada na pasta: {TEST_IMAGES_DIR}")
else:
    print(f"Total de imagens encontradas: {len(lista_completa_imagens)}")

    # Garanta que não tentemos selecionar mais imagens do que as disponíveis
    if len(lista_completa_imagens) < NUM_IMAGES_TO_SELECT:
        print(f"Aviso: O número de imagens encontradas ({len(lista_completa_imagens)}) é menor que o solicitado ({NUM_IMAGES_TO_SELECT}).")
        print("Usando todas as imagens disponíveis.")
        amostra_de_imagens = lista_completa_imagens
    else:
        # Selecione 50 imagens aleatoriamente da lista completa
        amostra_de_imagens = random.sample(lista_completa_imagens, NUM_IMAGES_TO_SELECT)

    print(f"Processando uma amostra aleatória de {len(amostra_de_imagens)} imagens...")

    # --- 3. Carregar o modelo ---
    model = YOLO(MODEL_PATH)

    # --- 4. Executar a predição na amostra de imagens ---
    # Agora, a 'source' é a nossa lista com os caminhos das imagens aleatórias
    model.predict(
        source=amostra_de_imagens,
        save=True,
        imgsz=640,
        conf=0.5,
        project=projeto_path,
        name=f'{treino_name}_img_test' # Um nome de pasta mais descritivo
    )

    print("\nPrevisão da amostra concluída!")
    print(f"As {len(amostra_de_imagens)} imagens com as detecções foram salvas na pasta: {projeto_path}/{treino_name}_img_test")

In [ ]:
from ultralytics import YOLO
import os
import shutil

!rm -r /content/exportados_rpi5 # apaga o diretorio caso ja exista.

dataset_paht = "/content/Abelhas-V3-1"
projeto_path = "/content/drive/MyDrive/Lesc-Bee videos/modelo_v2"
treino_name = "modelo_V2_09_02_2026"

# --- Configurações ---
original_model_path = f'{projeto_path}/{treino_name}3/weights/best.pt'

# 1. PREPARAÇÃO LOCAL
local_export_dir = '/content/exportados_rpi5'
os.makedirs(local_export_dir, exist_ok=True)
local_model_path = f'{local_export_dir}/best.pt'

print("--- Movendo modelo do Drive para o armazenamento local ---")
shutil.copy(original_model_path, local_model_path)
print("Modelo copiado para a máquina local com sucesso!")

imgsz = [640, 640]
data_yaml_path = f'{dataset_paht}/data.yaml'

print(f"\nCarregando modelo: {local_model_path}")
model = YOLO(local_model_path)

# Função auxiliar para evitar que os arquivos se sobrescrevam
def renomear_exportacao(caminho_antigo, novo_nome):
    novo_caminho = os.path.join(local_export_dir, novo_nome)
    # Se já existir de uma execução anterior, deleta para evitar erro
    if os.path.exists(novo_caminho):
        if os.path.isdir(novo_caminho):
            shutil.rmtree(novo_caminho)
        else:
            os.remove(novo_caminho)
    os.rename(caminho_antigo, novo_caminho)
    print(f"-> Salvo com o nome correto: {novo_nome}")

# ============================================
# 1. Exportar para ONNX (FP32)
# ============================================
print("\n--- Exportando para ONNX (FP32) ---")
onnx_path = model.export(format='onnx', imgsz=imgsz, simplify=True, opset=12)
renomear_exportacao(onnx_path, 'modelo_onnx_fp32.onnx')

# ============================================
# 2. Exportar para ONNX (FP16)
# ============================================
print("\n--- Exportando para ONNX (FP16) ---")
onnx_fp16_path = model.export(format='onnx', imgsz=imgsz, simplify=True, half=True, opset=12)
renomear_exportacao(onnx_fp16_path, 'modelo_onnx_fp16.onnx')

# ============================================
# 3. Exportar para OpenVINO (INT8)
# ============================================
print("\n--- Exportando para OpenVINO (INT8) ---")
openvino_int8_path = model.export(format='openvino', imgsz=imgsz, int8=True, data=data_yaml_path)
renomear_exportacao(openvino_int8_path, 'int8_openvino_model')

# ============================================
# 4. Exportar para OpenVINO (FP16)
# ============================================
print("\n--- Exportando para OpenVINO (FP16) ---")
openvino_path = model.export(format='openvino', imgsz=imgsz, half=True)
renomear_exportacao(openvino_path, 'fp16_openvino_model')

# ============================================
# 5. Exportar para OpenVINO (FP32)
# ============================================
print("\n--- Exportando para OpenVINO (FP32) ---")
openvino_path = model.export(format='openvino', imgsz=imgsz)
renomear_exportacao(openvino_path, 'fp32_openvino_model')


# ============================================
# 6. Exportar para NCNN (FP32)
# ============================================
print("\n--- Exportando para NCNN (FP32) ---")
ncnn_fp32_path = model.export(format='ncnn', imgsz=imgsz)
renomear_exportacao(ncnn_fp32_path, 'ncnn_fp32')

# ============================================
# 7. Exportar para NCNN (FP16)
# ============================================
print("\n--- Exportando para NCNN (FP16) ---")
ncnn_fp16_path = model.export(format='ncnn', imgsz=imgsz, half=True)
renomear_exportacao(ncnn_fp16_path, 'ncnn_fp16')

# ============================================
# 8. Compactar e Mover para o Drive
# ============================================
print("\n--- Compactando os modelos exportados ---")
zip_base_name = '/content/modelos_exportados_rpi5'

# Compacta tudo que está dentro do 'local_export_dir' (que agora tem os nomes certos!)
shutil.make_archive(zip_base_name, 'zip', local_export_dir)
print(f"Arquivo ZIP criado localmente em: {zip_base_name}.zip")

print("\n--- Movendo arquivo ZIP para o Google Drive ---")
drive_dest_dir = f'{projeto_path}/{treino_name}'
os.makedirs(drive_dest_dir, exist_ok=True)

drive_zip_path = f'{drive_dest_dir}/exportados_rpi5.zip'
shutil.copy(f'{zip_base_name}.zip', drive_zip_path)

print(f"\n✅ SUCESSO! Modelos gerados com nomes únicos e salvos no Drive em: {drive_zip_path}")

# ============================================
# 9. Preparação para Hailo-8L / Hailo-8 (Kit de IA)
# ============================================
print("\n--- NOTA SOBRE HAILO AI HAT+ ---")
print("Para compilar o arquivo '.hef' para o seu HAT de 13 ou 26 TOPS:")
print("Recomenda-se usar o modelo ONNX padrão (FP32) que você já exportou anteriormente.")
print("Alimente esse ONNX FP32 no Hailo Dataflow Compiler na sua máquina host.")
print("O compilador da Hailo fará a quantização nativa e otimizada para o chip deles.")

Teste dos modelos exportados com as imagens de teste.

In [ ]:
import os
import pandas as pd
from ultralytics import YOLO

dataset_paht = "/content/Abelhas-V3-1"
local_export_dir = '/content/exportados_rpi5'

# ==========================================
# 2. Definir os modelos com os nomes corretos
# ==========================================
modelos_para_testar = {
    "PyTorch_Original": f"{local_export_dir}/best.pt",
    "ONNX_FP32": f"{local_export_dir}/modelo_onnx_fp32.onnx",
    "ONNX_FP16": f"{local_export_dir}/modelo_onnx_fp16.onnx",
    "OpenVINO_FP32": f"{local_export_dir}/fp32_openvino_model",
    "OpenVINO_FP16": f"{local_export_dir}/fp16_openvino_model",
    "OpenVINO_INT8": f"{local_export_dir}/int8_openvino_model"
}

# ==========================================
# 3. Rodar os testes e extrair métricas
# ==========================================
resultados = []

for nome_modelo, caminho_modelo in modelos_para_testar.items():
    if not os.path.exists(caminho_modelo):
        print(f"⚠️ Pulando {nome_modelo}: Arquivo/Pasta não encontrado em {caminho_modelo}")
        continue

    print(f"\n========================================")
    print(f"🚀 Iniciando teste: {nome_modelo}")
    print(f"========================================")

    try:
        model = YOLO(caminho_modelo)

        metrics = model.val(
            data=f'{dataset_paht}/data.yaml',
            split='test',
            imgsz=640,
            plots=False,
            verbose=False
        )

        names = metrics.names
        map50_95 = metrics.box.maps
        precision = metrics.box.p
        recall = metrics.box.r
        map50_classes = metrics.box.all_ap[:, 0]
        map75_classes = metrics.box.all_ap[:, 5]

        for class_id, class_name in names.items():
            idx = int(class_id)
            resultados.append({
                "Modelo": nome_modelo,
                "Classe": class_name,
                "Precision": round(precision[idx], 4),
                "Recall": round(recall[idx], 4),
                "mAP50": round(map50_classes[idx], 4),
                "mAP75": round(map75_classes[idx], 4),
                "mAP50-95": round(map50_95[idx], 4)
            })

        print(f"✅ {nome_modelo} testado com sucesso!")

    except Exception as e:
        print(f"❌ Erro ao testar {nome_modelo}: {e}")


## Incluir o tempo de inferencia para cpu e gpu
# ==========================================
# 4. Gerar e Salvar o CSV
# ==========================================
print("\n--- Compilando resultados no CSV ---")
df_resultados = pd.DataFrame(resultados)
print(df_resultados.head(12))

caminho_csv = '/content/comparativo_modelos.csv'
df_resultados.to_csv(caminho_csv, index=False)

# Copia o arquivo para o drive
projeto_path = "/content/drive/MyDrive/Lesc-Bee videos/modelo_v2"
treino_name = "modelo_V2_09_02_2026"
shutil.copy(caminho_csv, f'{projeto_path}/{treino_name}/comparativo_modelos.csv')

# Cópia para o Drive para você não perder!
import shutil
projeto_path = "/content/drive/MyDrive/Lesc-Bee videos/modelo_v2"
treino_name = "modelo_V2_09_02_2026"
drive_csv_path = f'{projeto_path}/{treino_name}/comparativo_modelos.csv'
shutil.copy(caminho_csv, drive_csv_path)

print(f"\n✅ Tudo pronto! CSV salvo no seu Drive em: {drive_csv_path}")